#### [DT-ESP-C05: esp-c05 文檔](https://templates.blakadder.com/assets/DT-ESP-C05_Datasheet.pdf)

### table 2.2  
|Num|Pin Name|Type|Function|
|---|--------|----|--------|
|1  |IO0     |I/O |GPIO0, ADC1_CH0, XTAL_32K_P|
|2  |IO1     |I/O |GPIO1,ADC1_CH1, XTAL_32K_N|
|3  |IO2     |I/O |GPIO2,ADC1_CH2,**FSPIQ** |
|4  |EN      |I/O |         |
|5  |IO10    |I/O |GPIO10, **FSPICS0**|
|6  |IO9     |I/O |         |
|7  |IO3     |I/O |         |
|8  |IO4     |I/O |         |
|9  |IO5     |I/O |         |
|10 |IO6     |I/O |GPIO6,**FSPICLK**,MTCK |
|11 |IO7     |I/O |GPIO7,**FSPID**,MTDO |
|   |...     |    |         |


所以我們在 main.c define

```c
#define GPIO_MOSI 7   //FSPID          
#define GPIO_MISO 2   //FSPIQ       
#define GPIO_SCLK 6   //FSPICLK       
#define GPIO_CS 10    //FSPICS0 
```


注意標準流程是 先跑 esp32  
再在 rpi 跑 python test.py  (read c2_rpi_side_master.../readme.ipynb)

- [yt tutorial](https://www.youtube.com/watch?v=J-wr4fdvwBk&list=PLgrKXQgo8LPu_rr_19UcScZvbALLLJsjv&index=4)


#### SPI pins connection
```
Rpi master
 -------------     ------------
|SCLK(23=IO11)|-->|SCLK(IO6)   |
|MOSI(19=IO10)|-->|MOSI(IO7)   |
|MISO(21=IO9) |<--|MISO(IO2)   |
|CS  (24=IO8) |-->|CS  (IO10)  |
|GND (25=gnd) |---|GND         |
 -------------     ------------
```


### 正確的 rpi 連線   

```
                   RPi gpio pin  
             3.3 v---1  2 ---5v   
 (i2c)(SDA)GPIO  2---3  4 ---5v          
 (i2c)(SCL)GPIO  3---5  6 ---gnd         
           GPIO  4---7  8 ---GPIO 14 (TXD) (uart)
               gnd---9  10---GPIO 15 (RXD) (uart)
           GPIO 17---11 12---GPIO 18 
           GPIO 27---13 14---gnd          
           GPIO 22---15 16---gpio 23      
             3.3v ---17 18---gpio 24           
(spi)(MOSI)GPIO 10---19 20---gnd      
(spi)(MISO)GPIO  9---21 22---gpio 25     
(spi)(SCLK)GPIO 11---23 24---gpio 8 (CE0)(spi)          
               gnd---25 26---gpio 7 (CE1)(spi)      
           GPIO  0---27 28---gpio 1 
           GPIO  5---29 30---gnd 
           GPIO  6---31 32---gpio12
           GPIO 13---33 34---gnd
           GPIO 19---35 36---GPIO 16  
           GPIO 26---37 38---GPIO 20
              gnd ---39 40---GPIO 21 
```

### 正確的 esp32 連線


```
           IO0  ---1  2 ---IO18 
           IO1  ---3  4 ---IO19     
(SPI MISO) IO2  ---5  6 ---IO5        
(SPI CS)   IO10 ---7  8 ---IO6 (SPI SCLK)
           IO4  ---9  10---IO7 (SPI MOSI)
           IO5  ---11 12---3v3 
           gnd  ---13 14---gnd        
           gnd  ---15 16---gnd     
           gnd  ---17 18---gnd          
           3v3  ---19 20---gnd    
           3v3  ---21 22---IO9   
           gnd  ---23 24---RX         
           en   ---25 26---TX       
           gnd  ---27 28---gnd
           5v   ---29 30---3v3 
```

In [ ]:
#*** if you are not lazy
cd a_my_linux_device_driver_text_book/
cp -r ./d_esp_rpi_spi/ ~/esp/
cd ~/esp

In [ ]:
#*** dont run this in practice folder (build is large)  
#*** run this in ~/esp
cd ~/esp/d_esp_rpi_spi/
. $HOME/esp/esp-idf/export.sh  
idf.py set-target esp32c3
# idf.py menuconfig
# ----build the project----  
idf.py build
## write into esp32
idf.py -p /dev/ttyUSB0  flash 
##monitor the output  
# https://stackoverflow.com/questions/73923341/unable-to-flash-esp32-the-port-doesnt-exist
# idf.py -p /dev/ttyUSB0 monitor
# To exit IDF monitor use the shortcut Ctrl+].